# CPC353 Assignment 1 — Stop-Word Analysis

This notebook treats all included reviews as one combined Shin Ramyun corpus. It analyzes the project stop-word list while keeping the original wording and unfiltered tokens available.

Run preprocessing.ipynb first. The notebook saves two CSV tables in data/analysis/.


## 1. Imports and locate the project folder

The notebook searches the current folder and its parents for the preprocessing outputs, so it works when launched from the repository root or a subfolder.


In [1]:
import json
from collections import Counter
from pathlib import Path

import pandas as pd

PROJECT_DIR = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / "data" / "processed").is_dir()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError("Could not find data/processed. Run this notebook from the project.")

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
OUTPUT_DIR = PROJECT_DIR / "data" / "analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

try:
    from IPython.display import display
except ImportError:
    display = print

print(f"Project folder: {PROJECT_DIR}")
print(f"Analysis outputs: {OUTPUT_DIR}")

Project folder: C:\Users\USER\Desktop\Y3S1 CQ\CPC353\Assignment 1\CPC353-Assignment-1
Analysis outputs: C:\Users\USER\Desktop\Y3S1 CQ\CPC353\Assignment 1\CPC353-Assignment-1\data\analysis


## 2. Load the retained reviews and project stop words

All included collected and AI-generated reviews are combined before calculating the results. The analysis reports corpus-wide frequencies rather than separate origin totals. The review-language labels remain available for context, but are not used to split the main results.


In [2]:
def load_jsonl(path):
    with path.open(encoding="utf-8") as file:
        rows = [json.loads(line) for line in file if line.strip()]
    return [row for row in rows if row.get("included", True)]

reviews = (
    load_jsonl(PROCESSED_DIR / "real_preprocessed.jsonl")
    + load_jsonl(PROCESSED_DIR / "ai_preprocessed.jsonl")
)

stopwords_df = pd.read_csv(PROCESSED_DIR / "stopwords_used.csv")
stopwords_df["word"] = stopwords_df["word"].astype(str).str.casefold()
stopwords_df["language"] = stopwords_df["language"].astype(str)

stopword_language = dict(zip(stopwords_df["word"], stopwords_df["language"]))
stopword_set = set(stopword_language)

print(f"Included reviews in combined corpus: {len(reviews)}")
print(f"Project stop words loaded: {len(stopword_set)}")

Included reviews in combined corpus: 65
Project stop words loaded: 100


## 3. Corpus-wide stop-word frequencies

For every project stop word, the table records its count before filtering, how many occurrences were removed, and how many remain afterward. Counts are normalized per 1,000 unfiltered tokens to make the overall rate easy to interpret.

The preprocessing list is project-authored and conservative. Negators such as not, tak, and bukan are intentionally protected and will not appear as removed stop words.


In [3]:
def counts_for(rows, field):
    counts = Counter()
    for row in rows:
        counts.update(str(token).casefold() for token in row.get(field, []))
    return counts

before = counts_for(reviews, "tokens_before")
removed = counts_for(reviews, "tokens_removed")
after = counts_for(reviews, "tokens_after")
total_before = sum(before.values())

frequency_rows = []
for word in sorted(stopword_set):
    frequency_rows.append({
        "stopword_language": stopword_language[word],
        "word": word,
        "before_count": before[word],
        "removed_count": removed[word],
        "after_count": after[word],
        "before_per_1000_tokens": round(before[word] / total_before * 1000, 3)
            if total_before else 0,
        "reviews_containing_word": sum(
            word in {str(token).casefold() for token in row.get("tokens_before", [])}
            for row in reviews
        ),
        "total_unfiltered_tokens": total_before,
    })

stopword_frequency_df = pd.DataFrame(frequency_rows)
frequency_path = OUTPUT_DIR / "stopword_frequency_overall.csv"
stopword_frequency_df.to_csv(frequency_path, index=False, encoding="utf-8-sig")

print(f"Total unfiltered tokens: {total_before}")
print(f"Stop-word occurrences removed: {sum(removed.values())}")
print(f"Share of tokens removed: {sum(removed.values()) / total_before * 100:.2f}%")
print("Most frequent project stop words in the combined corpus:")
display(
    stopword_frequency_df.sort_values(
        ["before_count", "word"], ascending=[False, True]
    ).head(25)
)

print("Check: stop-word occurrences remaining after filtering:")
print(f"Total occurrences remaining: {sum(after[word] for word in stopword_set)}")

Total unfiltered tokens: 1201
Stop-word occurrences removed: 306
Share of tokens removed: 25.48%
Most frequent project stop words in the combined corpus:
   stopword_language    word  ...  reviews_containing_word  total_unfiltered_tokens
70           English     the  ...                       27                     1201
5            English     and  ...                       16                     1201
17             Malay     dan  ...                       20                     1201
65             Malay    saya  ...                       19                     1201
0            English       a  ...                       14                     1201
40           English       i  ...                       15                     1201
46           English      it  ...                        9                     1201
45           English      is  ...                        9                     1201
80           English      to  ...                        9                     1201
94    

## 4. Review-level stop-word use

This table shows how many stop-word occurrences each review had and what share of its unfiltered tokens they represented. It summarizes variation across the full corpus without grouping reviews by origin.


In [4]:
review_rows = []
for row in reviews:
    before_tokens = [str(token).casefold() for token in row.get("tokens_before", [])]
    removed_tokens = [str(token).casefold() for token in row.get("tokens_removed", [])]
    total_tokens = len(before_tokens)
    removed_english = sum(
        stopword_language.get(token) == "English" for token in removed_tokens
    )
    removed_malay = sum(
        stopword_language.get(token) == "Malay" for token in removed_tokens
    )
    review_rows.append({
        "review_id": row.get("review_id", ""),
        "review_language": row.get("language", ""),
        "unfiltered_token_count": total_tokens,
        "stopword_occurrences_before": sum(token in stopword_set for token in before_tokens),
        "stopword_occurrences_removed": len(removed_tokens),
        "english_stopword_occurrences_removed": removed_english,
        "malay_stopword_occurrences_removed": removed_malay,
        "removed_share_percent": round(len(removed_tokens) / total_tokens * 100, 2)
            if total_tokens else 0,
    })

review_stopword_df = pd.DataFrame(review_rows)
review_path = OUTPUT_DIR / "stopword_review_summary.csv"
review_stopword_df.to_csv(review_path, index=False, encoding="utf-8-sig")

print("Review-level stop-word summary (first 20 rows):")
display(review_stopword_df.head(20))
print("Overall mean percentage of tokens removed per review:")
print(f"{review_stopword_df['removed_share_percent'].mean():.2f}%")

Review-level stop-word summary (first 20 rows):
     review_id  ... removed_share_percent
0       REV-01  ...                 39.13
1       REV-02  ...                 40.00
2       REV-03  ...                 26.00
3       REV-04  ...                 42.31
4       REV-05  ...                 33.33
5       REV-06  ...                 33.33
6       REV-07  ...                 32.88
7       REV-08  ...                 23.53
8       REV-09  ...                  8.82
9       REV-10  ...                  6.25
10      REV-11  ...                 21.05
11      REV-12  ...                 41.67
12  REV-AI-001  ...                 14.29
13  REV-AI-002  ...                 14.29
14  REV-AI-003  ...                 29.41
15  REV-AI-004  ...                 31.25
16  REV-AI-005  ...                 15.38
17  REV-AI-006  ...                  9.09
18  REV-AI-007  ...                 15.38
19  REV-AI-008  ...                 20.00

[20 rows x 8 columns]
Overall mean percentage of tokens removed per r

## 5. Saved outputs and interpretation

The notebook creates:
- data/analysis/stopword_frequency_overall.csv
- data/analysis/stopword_review_summary.csv

The results combine the whole dataset. Use before_count to describe stop-word use, removed_count to confirm what the filter removed, and before_per_1000_tokens as a normalized frequency. The review summary shows whether stop words are spread across reviews or concentrated in a few. Keep the dataset's mixed real and AI-generated provenance clear in the report, but do not split the stop-word results by origin. These are CSV tables that Excel can open; they are not Excel workbooks.
